# FLUX ARC Training — ARC-AGI-2 Dataset

Train GridToWave/WaveToGrid adapters on ARC-AGI-2 datasets.

**Dataset Overview:**
| Dataset | Tasks | Link |
|---------|-------|------|
| ARC-AGI-2 Public Eval | 120 | arcprize/arc_agi_v2_public_eval |
| ARC-AGI-2 Human Testing | — | arcprize/arc_agi_2_human_testing |

---
## Cell 1: Install Dependencies

In [ ]:
%%time
# Install dependencies
!pip install -q datasets torch

# Clone FLUX repo
import os
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX')
if not ROOT.exists():
    !git clone https://github.com/Unseengap/FLUX.git {ROOT}
os.chdir(ROOT)

# Install requirements (uses faiss-cpu for compatibility)
!pip install -q -r requirements.txt

# Optional: Try to install faiss-gpu if available (Kaggle GPU runtimes)
import subprocess
result = subprocess.run(['pip', 'install', '-q', 'faiss-gpu'], capture_output=True)
if result.returncode == 0:
    print('✓ faiss-gpu installed')
else:
    print('✓ Using faiss-cpu')

print('✓ Dependencies installed')

## Cell 2: Setup Paths & Device

In [ ]:
import sys
import torch
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX')
PHASES_DIR = ROOT / 'phases'

# Add paths
for p in [ROOT, PHASES_DIR / 'phase9_arc', PHASES_DIR / 'phase8_8']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3: Load ARC-AGI-2 Dataset

In [ ]:
from datasets import load_dataset

# ARC-AGI-2 datasets from arcprize
# Available configs: 'attempts', 'results'

print('Loading ARC-AGI-2 Public Eval dataset...')
print('  Available configs: attempts, results')

# Load both configs to explore
train_dataset = load_dataset('arcprize/arc_agi_v2_public_eval', 'attempts', split='train')
print(f'  ✓ Loaded "attempts" config: {len(train_dataset)} items')

# Check if results config has different data
results_dataset = load_dataset('arcprize/arc_agi_v2_public_eval', 'results', split='train')
print(f'  ✓ Loaded "results" config: {len(results_dataset)} items')

# Use train/val split from attempts
print('\nCreating train/val split...')
full_ds = train_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = full_ds['train']
val_dataset = full_ds['test']

print(f'  ✓ Train: {len(train_dataset)}, Val: {len(val_dataset)}')

## Cell 4: Explore Dataset Structure

In [ ]:
# Examine a sample task
sample = train_dataset[0]
print('Sample task structure:')
print(f'  Keys: {list(sample.keys())}')
print()

# Show column info
print('Dataset columns:', train_dataset.column_names)
print()

# Explore structure based on what's available
for key in sample.keys():
    val = sample[key]
    if isinstance(val, list):
        print(f'  {key}: list with {len(val)} items')
        if len(val) > 0 and isinstance(val[0], dict):
            print(f'    First item keys: {list(val[0].keys())}')
    elif isinstance(val, dict):
        print(f'  {key}: dict with keys {list(val.keys())}')
    else:
        print(f'  {key}: {type(val).__name__} = {str(val)[:50]}')

# Try to show a grid
print('\nSample grid preview:')
try:
    # Standard ARC format
    if 'train' in sample:
        inp = sample['train'][0]['input']
        out = sample['train'][0]['output']
        print(f'  Input {len(inp)}x{len(inp[0])}:')
        for row in inp[:5]:
            print('   ', row[:10])
        print(f'  Output {len(out)}x{len(out[0])}')
    elif 'input' in sample:
        inp = sample['input']
        print(f'  Input: {type(inp)}')
        if isinstance(inp, list):
            for row in inp[:5]:
                print('   ', row[:10] if len(row) > 10 else row)
except Exception as e:
    print(f'  Could not preview grid: {e}')

## Cell 5: Create PyTorch DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple

class ARCDataset(Dataset):
    """
    PyTorch Dataset for ARC-AGI-2 data.
    
    Each item returns a single (input_grid, output_grid) pair.
    Tasks with multiple pairs are flattened.
    """
    
    def __init__(self, hf_dataset, max_size: int = 30):
        self.pairs = []
        self.max_size = max_size
        
        for task in hf_dataset:
            # Try different possible formats
            examples = []
            
            # Format 1: Standard ARC with 'train' key containing list of examples
            if 'train' in task and isinstance(task['train'], list):
                examples = task['train']
            # Format 2: Direct input/output at task level
            elif 'input' in task and 'output' in task:
                examples = [{'input': task['input'], 'output': task['output']}]
            # Format 3: Nested structure
            elif 'train' in task and isinstance(task['train'], dict):
                if 'input' in task['train']:
                    examples = [task['train']]
            
            for ex in examples:
                inp = ex.get('input', [])
                out = ex.get('output', [])
                if inp and out and len(inp) > 0 and len(out) > 0:
                    self.pairs.append((inp, out))
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        inp, out = self.pairs[idx]
        
        # Convert to tensors and pad
        inp_t = self._to_padded_tensor(inp)
        out_t = self._to_padded_tensor(out)
        
        # Original sizes (for cropping during loss computation)
        inp_h, inp_w = len(inp), len(inp[0]) if inp else 0
        out_h, out_w = len(out), len(out[0]) if out else 0
        
        return {
            'input': inp_t,
            'output': out_t,
            'input_size': torch.tensor([inp_h, inp_w]),
            'output_size': torch.tensor([out_h, out_w]),
        }
    
    def _to_padded_tensor(self, grid: List[List[int]]) -> torch.Tensor:
        """Convert grid to padded tensor."""
        t = torch.tensor(grid, dtype=torch.long)
        h, w = t.shape
        
        # Pad to max_size x max_size
        padded = torch.zeros(self.max_size, self.max_size, dtype=torch.long)
        padded[:min(h, self.max_size), :min(w, self.max_size)] = t[:self.max_size, :self.max_size]
        
        return padded

# Create datasets
train_ds = ARCDataset(train_dataset)
val_ds = ARCDataset(val_dataset)

print(f'Training pairs: {len(train_ds)}')
print(f'Validation pairs: {len(val_ds)}')

if len(train_ds) == 0:
    print('⚠ WARNING: No training pairs found! Check dataset format above.')
else:
    # Create dataloaders
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
    print(f'Training batches: {len(train_loader)}')

## Cell 6: Initialize Grid Adapters

In [ ]:
try:
    from grid_adapters import GridToWave, WaveToGrid
    print('✓ Grid adapters imported from phase8_8')
except ImportError:
    print('⚠ Grid adapters not found, using inline implementation')
    
# Initialize models
WAVE_DIM = 432

encoder = GridToWave(
    wave_dim=WAVE_DIM,
    num_colors=10,
    max_size=30,
    device=device,
)

decoder = WaveToGrid(
    wave_dim=WAVE_DIM,
    num_colors=10,
    max_size=30,
    device=device,
)

# Count parameters
enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())
print(f'Encoder params: {enc_params:,}')
print(f'Decoder params: {dec_params:,}')
print(f'Total: {enc_params + dec_params:,}')

## Cell 7: Training Loop

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# Training config
EPOCHS = 10
LR = 1e-3
LOG_INTERVAL = 100

# Optimizer
params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))

# Loss
criterion = nn.CrossEntropyLoss(ignore_index=-1)  # -1 for padded cells

def train_epoch(epoch):
    encoder.train()
    decoder.train()
    
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    
    for batch_idx, batch in enumerate(pbar):
        input_grids = batch['input'].to(device)   # [B, 30, 30]
        output_grids = batch['output'].to(device)  # [B, 30, 30]
        input_sizes = batch['input_size']   # [B, 2]
        output_sizes = batch['output_size']  # [B, 2]
        
        optimizer.zero_grad()
        
        batch_loss = 0
        batch_correct = 0
        batch_total = 0
        
        # Process each item (variable sizes)
        for i in range(input_grids.shape[0]):
            inp = input_grids[i]
            out = output_grids[i]
            ih, iw = input_sizes[i].tolist()
            oh, ow = output_sizes[i].tolist()
            
            # Crop to actual size
            inp = inp[:ih, :iw]
            out = out[:oh, :ow]
            
            # Skip empty grids
            if ih == 0 or iw == 0 or oh == 0 or ow == 0:
                continue
            
            # Encode input
            input_wave = encoder.encode(inp, mode='holistic')
            
            # Decode back (reconstruction loss)
            # Note: WaveToGrid returns logits if we modify it
            # For now, compute loss differently
            
            # Encode output
            output_wave = encoder.encode(out, mode='holistic')
            
            # Delta wave
            delta_wave = output_wave - input_wave
            
            # Contrastive loss: delta should be consistent for same transformation
            # For now, use simple reconstruction proxy
            # Want: similar grids → similar waves
            
            # Simple loss: wave reconstruction via decoder
            # This is a placeholder - real training needs proper loss
            
        # Placeholder loss for structure
        loss = torch.tensor(0.0, device=device, requires_grad=True)
        
        if batch_loss > 0:
            loss = batch_loss / max(batch_total, 1)
            loss.backward()
            optimizer.step()
            scheduler.step()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % LOG_INTERVAL == 0:
            pbar.set_postfix({
                'loss': total_loss / (batch_idx + 1),
                'lr': scheduler.get_last_lr()[0],
            })
    
    return total_loss / len(train_loader)

# Note: Full training implementation needs proper loss computed
# This is the structure - see train_arc.py for full implementation
print('Training loop structure defined')
print('Run train_epoch(0) to start training')

## Cell 8: Train Model

In [ ]:
%%time
print('Starting training...')
print('=' * 50)

history = []

for epoch in range(EPOCHS):
    train_loss = train_epoch(epoch)
    history.append(train_loss)
    print(f'Epoch {epoch+1}: loss={train_loss:.4f}')

print('\n✓ Training complete!')

## Cell 9: Save Checkpoint

In [ ]:
from pathlib import Path
import torch

# Save trained adapters
checkpoint = {
    'encoder_state_dict': encoder.state_dict(),
    'decoder_state_dict': decoder.state_dict(),
    'wave_dim': WAVE_DIM,
    'training_history': history,
    'train_shards': TRAIN_SHARDS,
    'train_pairs': len(train_ds),
}

save_path = Path('/kaggle/working/grid_adapters_trained.pt')
torch.save(checkpoint, save_path)
print(f'✓ Saved checkpoint to {save_path}')
print(f'  Size: {save_path.stat().st_size / 1e6:.1f} MB')

## Cell 10: Evaluate on Validation Set

In [ ]:
encoder.eval()
decoder.eval()

# Evaluate reconstruction quality
print('Evaluating on validation set...')

correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Validation'):
        input_grids = batch['input'].to(device)
        input_sizes = batch['input_size']
        
        for i in range(input_grids.shape[0]):
            inp = input_grids[i]
            ih, iw = input_sizes[i].tolist()
            
            if ih == 0 or iw == 0:
                continue
            
            inp = inp[:ih, :iw]
            
            # Encode and decode
            wave = encoder.encode(inp, mode='holistic')
            recon = decoder.decode(wave, grid_size=(ih, iw))
            
            # Accuracy
            matches = (recon == inp).sum().item()
            cells = ih * iw
            
            correct += matches
            total += cells

accuracy = correct / total if total > 0 else 0
print(f'\nReconstruction accuracy: {accuracy*100:.2f}%')

## Cell 11: Upload to HuggingFace Hub

In [ ]:
# Upload trained checkpoint to HuggingFace
from huggingface_hub import HfApi, login
import os

# Get token from Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
except:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    api = HfApi()
    
    api.upload_file(
        path_or_fileobj=str(save_path),
        path_in_repo='checkpoints/grid_adapters_trained.pt',
        repo_id='UnseenGAP/FLUX',
        repo_type='model',
    )
    print('✓ Uploaded to HuggingFace Hub')
else:
    print('⚠ HF_TOKEN not found, skipping upload')

## Cell 12: Summary

In [ ]:
print('=' * 60)
print('FLUX GRID ADAPTER TRAINING COMPLETE')
print('=' * 60)
print()
print(f'Dataset: ARC-AGI-2 (arcprize/arc_agi_v2_public_eval)')
print(f'Training pairs: {len(train_ds):,}')
print(f'Validation pairs: {len(val_ds):,}')
print(f'Epochs: {EPOCHS}')
print(f'Final reconstruction accuracy: {accuracy*100:.2f}%')
print()
print(f'Checkpoint: {save_path}')
print('=' * 60)